In [11]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch

In [12]:
# Série temporelle
data = np.array([1,2,3,4,5,6,7,8,9,10], dtype=float)

window_size = 4          # sequence_length
input_size = 1

def sliding_window(sequence, window_size):
    X, y = [], []
    for i in range(len(sequence) - window_size):
        X.append(sequence[i:i+window_size])   # fenêtre
        y.append(sequence[i+window_size])     # cible
    return np.array(X), np.array(y)

X, y = sliding_window(data, window_size)

print("Avant PyTorch :")
print("X shape =", X.shape)   # (batch_size, sequence_length)
print("y shape =", y.shape)   # (batch_size,)
print("\t")
print("Voici les sequences de X:\n",X,"\n")
print("Voici les sequences de y: \n",y)
# Conversion en tenseurs PyTorch
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.float32)

# Ajouter input_size = 1
X_torch = X_torch.unsqueeze(-1)   # (batch_size, seq_len, input_size)
y_torch = y_torch.unsqueeze(-1)   # (batch_size, 1)

print("\nAprès PyTorch :")
print("X_torch shape =", X_torch.shape)
print("y_torch shape =", y_torch.shape)


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x : (batch_size, sequence_length, input_size)
        out, (h_n, c_n) = self.lstm(x)
        # out : (batch_size, sequence_length, hidden_size)

        out = out[:, -1, :]     # (batch_size, hidden_size)
        out = self.fc(out)      # (batch_size, 1)
        return out


# Paramètres
input_size = 1
hidden_size = 16
learning_rate = 0.01
epochs = 200

model_lstm = LSTMModel(input_size, hidden_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=learning_rate)

# Entraînement
for epoch in range(epochs):
    optimizer.zero_grad()
    
    y_pred = model_lstm(X_torch)   # (batch_size, 1)
    loss = criterion(y_pred, y_torch)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 50 == 0:
        print(f"LSTM Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

print("\nExemple de prédiction LSTM :")
print("y réel :", y_torch[:3].squeeze())
print("y prédit :", model_lstm(X_torch)[:3].detach().squeeze())


Avant PyTorch :
X shape = (6, 4)
y shape = (6,)
	
Voici les sequences de X:
 [[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]
 [4. 5. 6. 7.]
 [5. 6. 7. 8.]
 [6. 7. 8. 9.]] 

Voici les sequences de y: 
 [ 5.  6.  7.  8.  9. 10.]

Après PyTorch :
X_torch shape = torch.Size([6, 4, 1])
y_torch shape = torch.Size([6, 1])
LSTM Epoch [50/200], Loss: 2.8333
LSTM Epoch [100/200], Loss: 0.6663
LSTM Epoch [150/200], Loss: 0.0313
LSTM Epoch [200/200], Loss: 0.0023

Exemple de prédiction LSTM :
y réel : tensor([5., 6., 7.])
y prédit : tensor([4.9931, 5.9939, 7.0271])
